# Práctica 2 — Optimización Estocástica

**Inversión multietapa con datos de mercado**

Universidad EAFIT · Maestría en Matemáticas Aplicadas · 2026-2

Se calibra un árbol de escenarios de tres trimestres con cierres ajustados de
SPY, LQD y GLD (2010–2025), se resuelve el programa lineal multietapa con
**L-shaped anidado** y se aplica la política obtenida a los rendimientos
observados entre enero y septiembre de 2026.

**Notas de implementación.**

- Un único notebook autocontenido: no escribe módulos, todo el código vive en
  estas celdas y se ejecuta con *Run all*.
- Solver: `scipy.optimize.linprog` (HiGHS). No se usa Gurobi, así que no hay
  licencias de por medio. Todos los LP del algoritmo (maestros de cada nodo y
  problemas terminales) **se resuelven con el solver** y los multiplicadores
  duales se leen de su salida; la forma cerrada del problema terminal solo se
  usa como verificación independiente.
- Datos: únicamente `yfinance` (Yahoo Finance, campo *Adj Close*).
- No hay componentes aleatorios: no se usan semillas.
- Unidades: miles de dólares.

## 0. Preparación del entorno

In [ ]:
%pip install -q yfinance scipy pandas matplotlib

In [ ]:
import itertools
import json
import os
import time
import zipfile
from collections import Counter
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import display
from matplotlib.patches import FancyBboxPatch
from scipy.optimize import linprog

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

# --- Parámetros del problema (sección 3.1 del enunciado), en miles de USD ---
ACTIVOS = ["SPY", "LQD", "GLD"]
B0 = 100.0        # capital inicial b
META = 106.0      # meta terminal M
LAMBDA = 4.0      # penalización por cada unidad de déficit
TOL_META = 1e-6   # tolerancia monetaria (miles de USD) para "alcanza la meta"

# --- Parámetros del algoritmo ---
TOL_LSHAPED = 1e-6        # (UB-LB)/max{1,|UB|,|LB|}
TIEMPO_LIMITE = 3600.0    # segundos, por si algo se atasca
MAX_CICLOS = 5000

# --- Datos ---
DESCARGAR = True   # False: lee los CSV de cierres trimestrales ya guardados en datos/
DIR_DATOS, DIR_RES, DIR_FIG = "datos", "resultados", "figuras"
for _d in (DIR_DATOS, DIR_RES, DIR_FIG):
    os.makedirs(_d, exist_ok=True)

print("pandas", pd.__version__, "| yfinance", yf.__version__)

# Formulación matemática

## Conjuntos, estados y árbol

- Activos $I=\{\text{SPY},\text{LQD},\text{GLD}\}$ y estados conjuntos
  $U=\{AAA,AAB,\dots,BBB\}$ ($A$: rendimiento $\ge 0$; $B$: rendimiento $<0$;
  orden SPY, LQD, GLD).
- Probabilidad de cada estado $\hat q_u=n_u/64$ y factor representativo
  $\hat R_i(u)=\frac1{n_u}\sum_{j\in J_u}R_{i,j}$. Estados de trimestres distintos
  independientes: $P(z_1,z_2,z_3)=\hat q_{z_1}\hat q_{z_2}\hat q_{z_3}$.
- Etapas $t=1,2,3$ (decisión) y $t=4$ (evaluación terminal), con $K_t=8^{t-1}$
  nodos: $1,8,64,512$. Nodo $(t,k)$, padre $a(k)$, hijos $C_{t+1}(k)$,
  probabilidad acumulada $p^t_k$ y condicional $q_{\ell|k}=p^{t+1}_\ell/p^t_k$.

## LP nodal (equivalente determinista)

$$\min\ \sum_{k=1}^{512}p^4_k\big(\lambda w^4_k-y^4_k\big)$$

$$\begin{aligned}
&\textstyle\sum_i x^1_{i,1}=b \\
&\textstyle\sum_i \hat R_i(u_k)\,x^{t-1}_{i,a(k)}-\sum_i x^t_{i,k}=0,\quad t=2,3,\ k=1,\dots,K_t\\
&\textstyle\sum_i \hat R_i(u_k)\,x^{3}_{i,a(k)}-y^4_k+w^4_k=M,\quad k=1,\dots,512\\
&x^t_{i,k}\ge 0,\quad y^4_k,w^4_k\ge 0
\end{aligned}$$

Forma de clase: $T_{t-1,k}x^{t-1}_{a(k)}+W_tx^t_k=h^t_k$ con
$T_{t-1,k}=\hat R(u_k)^\top$, $W_1=\mathbf 1^\top$, $W_t=-\mathbf 1^\top$ ($t=2,3$),
$W_4=(-1,\ 1)$ sobre $(y,w)$, $h^1=b$, $h^t=0$ ($t=2,3$), $h^4=M$.

## Cota inicial de $\theta$

$R_{\max}=\max\{1,\max_{i,u}\hat R_i(u)\}$ y $L=\min\{0,\,M-bR_{\max}^3\}$. Toda
riqueza terminal alcanzable es $\le bR_{\max}^3$ y el costo terminal
$\lambda w-y\ge M-V$ (porque $\lambda\ge1$), de modo que todo costo futuro
esperado es $\ge L$. Se inicia $\theta^t_k\ge L$ (con $\theta$ libre de ser negativa).

## Cortes y algoritmo

Para un padre $(t,k)$ fijado en $\bar x^t_k$ y un hijo $(t+1,\ell)$ con valor óptimo
local $v^{t+1}_\ell(\bar x^t_k)$ y dual $\pi^{t+1}_\ell$ del balance:

$$\beta^{t+1}_\ell=-T_{t,\ell}^\top\pi^{t+1}_\ell,\qquad
\alpha^{t+1}_\ell=v^{t+1}_\ell(\bar x^t_k)-(\beta^{t+1}_\ell)^\top\bar x^t_k,$$

$$\theta^t_k\ \ge\ \sum_{\ell\in C_{t+1}(k)}q_{\ell|k}\big[\alpha^{t+1}_\ell+(\beta^{t+1}_\ell)^\top x^t_k\big].$$

Ciclo: (i) *forward*: resolver raíz y descendientes fijando la decisión del padre
y guardar la política completa; (ii) *backward*: de $t=3$ a $t=1$, generar el corte
de cada nodo con sus hijos (si un hijo intermedio recibió cortes nuevos se resuelve
de nuevo en el punto heredado antes de usar sus duales). $LB$ es el objetivo del
maestro raíz y $UB$ el menor costo esperado de las políticas completas evaluado en
las 512 hojas. Parada: $(UB-LB)/\max\{1,|UB|,|LB|\}\le10^{-6}$.

## 1. Datos: cierres ajustados con `yfinance`

Dos descargas diarias (campo **Adj Close**, `auto_adjust=False`):

| Bloque | Ventana pedida | Se conserva |
|---|---|---|
| Calibración | 2009-12-01 a 2025-12-31 (fin exclusivo 2026-01-01) | último cierre de dic-2009 y de cada trimestre 2010–2025: **65 precios** |
| Validación | 2025-12-01 a 2026-09-30 (fin exclusivo 2026-10-01) | últimos cierres de dic-2025, mar, jun y sep-2026: **4 precios** |

El último día de negociación de cada trimestre se obtiene como la última fila
disponible del trimestre calendario (si el último día calendario es festivo o fin
de semana, es la jornada bursátil anterior). El cierre de septiembre de 2026
**solo se acepta si el trimestre ya terminó**: se verifica la fecha del
último dato y la fecha actual.

In [ ]:
INICIO_CAL, FIN_CAL = "2009-12-01", "2026-01-01"   # fin exclusivo
INICIO_VAL, FIN_VAL = "2025-12-01", "2026-10-01"   # fin exclusivo


def descargar_adj_close(tickers, inicio, fin_exclusivo, intentos=3, pausa=5):
    """Descarga cierres diarios ajustados (Adj Close) con yfinance.

    auto_adjust=False para conservar explícitamente la columna 'Adj Close'
    (el cierre ajustado por dividendos y particiones) y no sumar nada dos veces.
    """
    ultimo_error = None
    for i in range(intentos):
        try:
            d = yf.download(tickers, start=inicio, end=fin_exclusivo, interval="1d",
                            auto_adjust=False, actions=False, progress=False,
                            threads=False)
            adj = d["Adj Close"][tickers].copy()
            adj.index = pd.DatetimeIndex(adj.index).tz_localize(None).normalize()
            adj = adj.sort_index()
            if adj.dropna(how="any").empty:
                raise ValueError("descarga vacía")
            return adj
        except Exception as e:  # reintento: la interfaz de Yahoo puede limitar consultas
            ultimo_error = e
            print(f"  intento {i + 1}/{intentos} falló: {e!r}")
            time.sleep(pausa)
    raise RuntimeError(f"No se pudo descargar {tickers}: {ultimo_error!r}")


def cierres_trimestrales(adj):
    """Último cierre ajustado de cada trimestre calendario y su fecha."""
    adj = adj.dropna(how="any").sort_index()
    q = adj.index.to_period("Q")
    cierres = adj.groupby(q).last()
    fechas = pd.Series(adj.index, index=adj.index).groupby(q).last()
    out = cierres.copy()
    out.insert(0, "fecha_cierre", fechas.dt.normalize())
    out.index = [str(p) for p in out.index]          # '2010Q1', ...
    out.index.name = "trimestre"
    return out


def validar_cierres(df, primero, ultimo, n_esperado, nombre, tol_dias=5):
    """Controles de calidad de una tabla de cierres trimestrales."""
    esperado = [str(p) for p in pd.period_range(primero, ultimo, freq="Q")]
    assert list(df.index) == esperado, f"{nombre}: trimestres {list(df.index)} != {esperado}"
    assert len(df) == n_esperado, f"{nombre}: {len(df)} precios, se esperaban {n_esperado}"
    assert (df[ACTIVOS] > 0).all().all(), f"{nombre}: precios no positivos"
    for tri, fila in df.iterrows():
        fin = pd.Period(tri, freq="Q").end_time.normalize()
        brecha = (fin - fila["fecha_cierre"]).days
        assert 0 <= brecha <= tol_dias, (
            f"{nombre}: el último dato de {tri} es {fila['fecha_cierre'].date()}, "
            f"a {brecha} días del fin del trimestre (¿trimestre incompleto?)")
    print(f"{nombre}: OK ({len(df)} precios por activo, "
          f"{df.index[0]} a {df.index[-1]}, último dato {df['fecha_cierre'].iloc[-1].date()})")


ruta_cal = os.path.join(DIR_DATOS, "cierres_calibracion.csv")
ruta_val = os.path.join(DIR_DATOS, "cierres_validacion.csv")
ruta_meta = os.path.join(DIR_DATOS, "metadatos_descarga.json")

if DESCARGAR:
    ahora = pd.Timestamp.now()
    assert ahora >= pd.Timestamp("2026-10-01"), (
        "El trimestre julio-septiembre de 2026 aún no termina: no use un cierre parcial.")
    print("Descargando calibración ...")
    adj_cal = descargar_adj_close(ACTIVOS, INICIO_CAL, FIN_CAL)
    print("Descargando validación ...")
    adj_val = descargar_adj_close(ACTIVOS, INICIO_VAL, FIN_VAL)
    cal = cierres_trimestrales(adj_cal).loc["2009Q4":"2025Q4"]
    val = cierres_trimestrales(adj_val).loc["2025Q4":"2026Q3"]
    validar_cierres(cal, "2009Q4", "2025Q4", 65, "Calibración")
    validar_cierres(val, "2025Q4", "2026Q3", 4, "Validación")
    cal.to_csv(ruta_cal)
    val.to_csv(ruta_val)
    with open(ruta_meta, "w", encoding="utf-8") as f:
        json.dump({
            "fuente": "Yahoo Finance, a través de la librería yfinance",
            "version_yfinance": yf.__version__,
            "campo": "Adj Close (auto_adjust=False), frecuencia diaria",
            "activos": ACTIVOS,
            "ventana_calibracion": [INICIO_CAL, FIN_CAL + " (exclusivo)"],
            "ventana_validacion": [INICIO_VAL, FIN_VAL + " (exclusivo)"],
            "fecha_consulta_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        }, f, indent=2, ensure_ascii=False)
else:
    cal = pd.read_csv(ruta_cal, index_col="trimestre", parse_dates=["fecha_cierre"])
    val = pd.read_csv(ruta_val, index_col="trimestre", parse_dates=["fecha_cierre"])
    validar_cierres(cal, "2009Q4", "2025Q4", 65, "Calibración")
    validar_cierres(val, "2025Q4", "2026Q3", 4, "Validación")

with open(ruta_meta, encoding="utf-8") as f:
    META_DESCARGA = json.load(f)
print("Fecha de consulta (UTC):", META_DESCARGA["fecha_consulta_utc"])
display(pd.concat([cal.head(3), cal.tail(2)]))
display(val)

## 2. Rendimientos trimestrales y estadísticas (solo calibración)

$r_{i,j}=P_{i,j}/P_{i,j-1}-1$ y $R_{i,j}=1+r_{i,j}$, $j=1,\dots,64$.

In [ ]:
P_cal = cal[ACTIVOS].to_numpy(dtype=float)           # (65, 3)
R_hist = P_cal[1:] / P_cal[:-1]                      # factores brutos (64, 3)
r_hist = R_hist - 1.0                                # rendimientos simples (64, 3)
N_HIST = r_hist.shape[0]
assert N_HIST == 64

est = pd.DataFrame({
    "media": r_hist.mean(axis=0), "desv_est": r_hist.std(axis=0, ddof=1),
    "min": r_hist.min(axis=0), "max": r_hist.max(axis=0)}, index=ACTIVOS)
print("Estadísticas de los 64 rendimientos trimestrales (en %)")
display((est * 100).round(3))

In [ ]:
# Gráfica: precios ajustados normalizados a 100 en diciembre de 2009
fig, ax = plt.subplots(figsize=(8, 4))
fechas_cal = cal["fecha_cierre"]
for i, a in enumerate(ACTIVOS):
    ax.plot(fechas_cal, 100 * P_cal[:, i] / P_cal[0, i], label=a, linewidth=1.8)
ax.set_title("Cierres ajustados trimestrales (base 100 = dic. 2009)")
ax.set_ylabel("índice")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(DIR_FIG, "fig_precios.png"), dpi=200)
plt.show()

## 3. Estados conjuntos, probabilidades y factores

Terna de signos en el orden SPY, LQD, GLD: `B` si el rendimiento es negativo,
`A` en caso contrario. La probabilidad de cada estado se estima contando
coincidencias de los **tres** signos en un mismo trimestre (no se multiplican
frecuencias marginales).

In [ ]:
ESTADOS_TODOS = ["".join(c) for c in itertools.product("AB", repeat=3)]
# ['AAA','AAB','ABA','ABB','BAA','BAB','BBA','BBB']


def terna(r):
    """Terna de signos de un vector de tres rendimientos."""
    return "".join("B" if x < 0 else "A" for x in r)


z_hist = [terna(r) for r in r_hist]
conteos = Counter(z_hist)

filas = []
for u in ESTADOS_TODOS:
    n_u = conteos.get(u, 0)
    idx = [j for j, z in enumerate(z_hist) if z == u]
    Rb = R_hist[idx].mean(axis=0) if n_u > 0 else np.full(3, np.nan)
    filas.append({"estado": u, "n_u": n_u, "q_u": n_u / N_HIST,
                  **{f"R_{a}": Rb[i] for i, a in enumerate(ACTIVOS)}})
tabla_estados = pd.DataFrame(filas)
display(tabla_estados.round(5))

assert tabla_estados["n_u"].sum() == N_HIST
vacios = tabla_estados.loc[tabla_estados["n_u"] == 0, "estado"].tolist()
print("Estados sin observaciones:", vacios if vacios else "ninguno (los 8 aparecen)")

# Estados con soporte: solo estos forman las ramas del árbol. Si hubiera un grupo
# vacío se elimina la rama (sin inventarle probabilidad ni promedio) y se activa el
# respaldo de la sección 5.2 en la validación.
ESTADOS = tabla_estados.loc[tabla_estados["n_u"] > 0, "estado"].tolist()
Q_ESTADO = tabla_estados.loc[tabla_estados["n_u"] > 0, "q_u"].to_numpy()
R_ESTADO = tabla_estados.loc[tabla_estados["n_u"] > 0,
                             [f"R_{a}" for a in ACTIVOS]].to_numpy()
assert abs(Q_ESTADO.sum() - 1.0) < 1e-12

## 4. Árbol de escenarios

Nodo $(t,k)$ con $k=1,\dots,K_t$, $K_t=S^{t-1}$ ($S$ ramas por nodo; $S=8$ en el
modelo completo). Los hijos del nodo $k$ de la etapa $t$ son $k\cdot S+s$,
$s=0,\dots,S-1$ (indexación interna base 0; los archivos exportados usan base 1).
El orden de $s$ es el de la lista de estados, por lo que la historia
$(z_1,\dots,z_{t-1})$ es el número en base $S$ de $k$.

In [ ]:
class Arbol:
    """Árbol de escenarios de 4 etapas con S ramas por nodo (independencia temporal)."""

    def __init__(self, estados, q, Rhat):
        self.estados = list(estados)
        self.S = len(self.estados)
        self.q = np.asarray(q, dtype=float)
        self.Rhat = np.asarray(Rhat, dtype=float)               # (S, 3)
        assert abs(self.q.sum() - 1.0) < 1e-12
        S = self.S
        self.K = {t: S ** (t - 1) for t in (1, 2, 3, 4)}
        self.padre, self.rama, self.p, self.R_rama = {}, {}, {1: np.array([1.0])}, {}
        for t in (2, 3, 4):
            k = np.arange(self.K[t])
            self.padre[t] = k // S                              # a(k)
            self.rama[t] = k % S                                # estado de la rama que llega a k
            self.p[t] = self.p[t - 1][self.padre[t]] * self.q[self.rama[t]]
            self.R_rama[t] = self.Rhat[self.rama[t]]            # (K_t, 3) = T_{t-1,k}^T
        assert abs(self.p[4].sum() - 1.0) < 1e-12

    def hijos(self, t, k):
        return range(k * self.S, (k + 1) * self.S)

    def q_cond(self, t, k):
        """Probabilidad condicional de llegar al nodo (t,k) desde su padre."""
        return self.p[t][k] / self.p[t - 1][self.padre[t][k]]

    def historia(self, t, k):
        """Historia (z_1..z_{t-1}) del nodo, como texto 'AAA-ABA'."""
        if t == 1:
            return "raíz"
        dig = []
        for _ in range(t - 1):
            dig.append(k % self.S)
            k //= self.S
        return "-".join(self.estados[d] for d in reversed(dig))

    def tabla_nodos(self):
        filas = []
        for t in (1, 2, 3, 4):
            for k in range(self.K[t]):
                filas.append({
                    "etapa": t, "k": k + 1,
                    "tipo": "hoja" if t == 4 else "inversion",
                    "padre": "" if t == 1 else int(self.padre[t][k]) + 1,
                    "historia": self.historia(t, k),
                    "estado_rama": "" if t == 1 else self.estados[self.rama[t][k]],
                    "prob_condicional": 1.0 if t == 1 else self.q_cond(t, k),
                    "prob_acumulada": self.p[t][k]})
        return pd.DataFrame(filas)


arbol = Arbol(ESTADOS, Q_ESTADO, R_ESTADO)
print("Nodos por etapa:", arbol.K, "| total:", sum(arbol.K.values()))
tab_nodos = arbol.tabla_nodos()
tab_nodos.to_csv(os.path.join(DIR_RES, "arbol_nodos.csv"), index=False)

# Controles: 512 probabilidades terminales suman 1 y las condicionales de cada padre también
print("Suma de las probabilidades terminales:", arbol.p[4].sum())
assert abs(arbol.p[4].sum() - 1.0) < 1e-12
for t in (1, 2, 3):
    for k in range(arbol.K[t]):
        assert abs(sum(arbol.q_cond(t + 1, l) for l in arbol.hijos(t, k)) - 1.0) < 1e-12
print("Probabilidades condicionales de cada padre suman 1: OK")
display(tab_nodos.groupby("etapa").size().rename("nodos").to_frame().T)

In [ ]:
# Figura: esquema por niveles y detalle de las ramas de la raíz y del nodo (2,1)
def figura_arbol(arbol):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6),
                             gridspec_kw={"width_ratios": [1.05, 1, 1]})
    ax = axes[0]
    ax.axis("off")
    ax.set_xlim(-0.2, 9.4)
    ax.set_ylim(-0.2, 1.4)
    nombres = [("Etapa 1\n(raíz)", arbol.K[1]), ("Etapa 2", arbol.K[2]),
               ("Etapa 3", arbol.K[3]), ("Etapa 4\n(hojas)", arbol.K[4])]
    for j, (nm, n) in enumerate(nombres):
        x0 = 0.1 + j * 2.35
        color = "#cfe3f5" if j < 3 else "#f5dcc2"
        ax.add_patch(FancyBboxPatch((x0, 0.35), 1.7, 0.6, boxstyle="round,pad=0.03",
                                    fc=color, ec="#355c7d"))
        ax.text(x0 + 0.85, 0.65, f"{n}\n{'nodo' if n == 1 else 'nodos' if j < 3 else 'hojas'}",
                ha="center", va="center", fontsize=9)
        ax.text(x0 + 0.85, 1.12, nm, ha="center", va="center", fontsize=9)
        if j < 3:
            ax.annotate("", xy=(x0 + 2.3, 0.65), xytext=(x0 + 1.75, 0.65),
                        arrowprops=dict(arrowstyle="->", color="#355c7d"))
            ax.text(x0 + 2.03, 0.78, f"×{arbol.S}", ha="center", fontsize=8)
    ax.set_title("Niveles del árbol", fontsize=10)

    def abanico(ax, titulo, raiz_txt, ramas, col_derecha):
        ax.axis("off")
        ys = np.linspace(len(ramas) - 1, 0, len(ramas))
        ax.scatter([0], [ys.mean()], s=420, color="#355c7d", zorder=3)
        ax.text(0, ys.mean(), raiz_txt, color="white", ha="center", va="center", fontsize=7, zorder=4)
        for y, (etq, val) in zip(ys, ramas):
            ax.plot([0, 1], [ys.mean(), y], color="#7f9db8", lw=1)
            ax.scatter([1], [y], s=60, color="#f0a35c", zorder=3)
            ax.text(1.06, y, f"{etq}   {col_derecha} = {val:.4f}", va="center", fontsize=8)
        ax.set_xlim(-0.3, 2.6)
        ax.set_ylim(-0.7, len(ramas) - 0.3)
        ax.set_title(titulo, fontsize=10)

    ramas_raiz = [(arbol.estados[s], arbol.q[s]) for s in range(arbol.S)]
    abanico(axes[1], "Ocho ramas de la raíz (1,1)", "(1,1)", ramas_raiz, r"$\hat q$")
    ramas_21 = [(arbol.historia(3, s), arbol.p[3][s]) for s in arbol.hijos(2, 0)]
    abanico(axes[2], f"Ocho ramas del nodo (2,1), historia {arbol.historia(2, 0)}",
            "(2,1)", ramas_21, r"$p^3_k$")
    fig.tight_layout()
    return fig


fig = figura_arbol(arbol)
fig.savefig(os.path.join(DIR_FIG, "fig_arbol.png"), dpi=200, bbox_inches="tight")
plt.show()

## 5. Modelo: LP nodal, equivalente determinista y L-shaped anidado

### 5.1 Equivalente determinista (verificación)

Se escribe todo el árbol en un único LP con `linprog`. Para el árbol completo debe
tener $3(1+8+64)+2(512)=1243$ variables y $1+8+64+512=585$ igualdades; para el
árbol reducido de dos estados, 37 variables y 15 igualdades. **No** sustituye al
L-shaped anidado: se usa para verificarlo (instancia reducida) y como comparación
adicional en la instancia completa.

In [ ]:
OPCIONES_HIGHS = {"primal_feasibility_tolerance": 1e-10,
                  "dual_feasibility_tolerance": 1e-10}


def equivalente_determinista(arbol, b=B0, M=META, lam=LAMBDA):
    """LP único con todos los nodos del árbol (sin sustituir factores por promedios)."""
    K, off, n = arbol.K, {}, 0
    for t in (1, 2, 3):
        off[t] = n
        n += 3 * K[t]                  # x^t_{i,k} en off[t] + 3k + i
    off[4] = n
    n += 2 * K[4]                      # (y_k, w_k) en off[4] + 2k, off[4] + 2k + 1

    filas, cols, vals, rhs = [], [], [], []
    r = 0
    for i in range(3):                                    # raíz: W_1 x = b
        filas.append(r); cols.append(off[1] + i); vals.append(1.0)
    rhs.append(b); r += 1
    for t in (2, 3):                                      # T x_padre + W x = 0
        for k in range(K[t]):
            a = arbol.padre[t][k]
            for i in range(3):
                filas.append(r); cols.append(off[t - 1] + 3 * a + i)
                vals.append(arbol.R_rama[t][k, i])
                filas.append(r); cols.append(off[t] + 3 * k + i); vals.append(-1.0)
            rhs.append(0.0); r += 1
    for k in range(K[4]):                                 # hojas: V - y + w = M
        a = arbol.padre[4][k]
        for i in range(3):
            filas.append(r); cols.append(off[3] + 3 * a + i)
            vals.append(arbol.R_rama[4][k, i])
        filas.append(r); cols.append(off[4] + 2 * k); vals.append(-1.0)
        filas.append(r); cols.append(off[4] + 2 * k + 1); vals.append(1.0)
        rhs.append(M); r += 1

    from scipy import sparse
    A_eq = sparse.csr_matrix((vals, (filas, cols)), shape=(r, n))
    c = np.zeros(n)
    for k in range(K[4]):
        c[off[4] + 2 * k] = -arbol.p[4][k]
        c[off[4] + 2 * k + 1] = lam * arbol.p[4][k]

    t0 = time.time()
    res = linprog(c, A_eq=A_eq, b_eq=np.array(rhs), bounds=(0, None),
                  method="highs", options=OPCIONES_HIGHS)
    assert res.status == 0, res.message
    X = {t: res.x[off[t]:off[t] + 3 * K[t]].reshape(K[t], 3) for t in (1, 2, 3)}
    Y = res.x[off[4]:off[4] + 2 * K[4]:2]
    W = res.x[off[4] + 1:off[4] + 2 * K[4]:2]
    return {"obj": float(res.fun), "X": X, "Y": Y, "W": W, "n_vars": n, "n_eq": r,
            "tiempo": time.time() - t0}


de_full_tamano = equivalente_determinista(arbol)
print(f"Equivalente determinista completo: {de_full_tamano['n_vars']} variables, "
      f"{de_full_tamano['n_eq']} igualdades, objetivo = {de_full_tamano['obj']:.8f} "
      f"({de_full_tamano['tiempo']:.2f} s)")
assert (de_full_tamano["n_vars"], de_full_tamano["n_eq"]) == (1243, 585) or arbol.S != 8

### 5.2 L-shaped anidado

Cada nodo no terminal $(t,k)$, $t=1,2,3$, tiene un maestro LP local

$$\min_{x^t_k,\theta^t_k}\ \theta^t_k\quad\text{s.a.}\quad
T_{t-1,k}x^{t-1}_{a(k)}+W_tx^t_k=h^t_k,\ \ \theta^t_k\ge\alpha_m+\beta_m^\top x^t_k\ (m=1,\dots),\ \ x^t_k\ge0,\ \theta^t_k\ge L$$

(sin costo inmediato). Cada hoja resuelve $\min\{\lambda w-y:\ \hat R(u)^\top\bar x-y+w=M,\ y,w\ge0\}$,
sin $\theta$. El dual de la igualdad de balance, $\pi$, se lee de
`res.eqlin.marginals` (sensibilidad del óptimo al lado derecho
$h-T\bar x$; se verifica numéricamente más abajo).

In [ ]:
class LShapedAnidado:
    """L-shaped anidado con recorridos completos del árbol (forward / backward)."""

    def __init__(self, arbol, b=B0, M=META, lam=LAMBDA, tol=TOL_LSHAPED,
                 tiempo_limite=TIEMPO_LIMITE, max_ciclos=MAX_CICLOS, tol_corte=1e-9,
                 nodos_registro=((1, 0), (2, 0), (3, 0)), verbose=True, nombre=""):
        self.arbol, self.b, self.M, self.lam = arbol, b, M, lam
        self.tol, self.tiempo_limite, self.max_ciclos = tol, tiempo_limite, max_ciclos
        self.tol_corte, self.verbose, self.nombre = tol_corte, verbose, nombre
        self.nodos_registro = set(nodos_registro)
        # Cota inicial de theta: L = min{0, M - b Rmax^3}
        self.Rmax = max(1.0, float(arbol.Rhat.max()))
        self.L = min(0.0, M - b * self.Rmax ** 3)
        # Probabilidades condicionales q_{l|k} por etapa
        self.qc = {t: arbol.p[t] / arbol.p[t - 1][arbol.padre[t]] for t in (2, 3, 4)}
        # Cortes por nodo: listas de (alpha, beta); la matriz se arma de forma perezosa
        self.cortes = {t: [([], []) for _ in range(arbol.K[t])] for t in (1, 2, 3)}
        self._cache = {}
        self.registro_cortes = {nk: [] for nk in self.nodos_registro}
        self.n_lp = 0
        self._bounds_nodo = [(0, None)] * 3 + [(self.L, None)]
        self._c_nodo = np.array([0.0, 0.0, 0.0, 1.0])
        self._A_eq_raiz = np.array([[1.0, 1.0, 1.0, 0.0]])          # W_1
        self._A_eq_nodo = np.array([[-1.0, -1.0, -1.0, 0.0]])       # W_t, t = 2, 3

    # ---- utilidades ----------------------------------------------------
    def n_cortes(self):
        return sum(len(a) for t in self.cortes for a, _ in self.cortes[t])

    def _lp(self, *args, **kw):
        self.n_lp += 1
        res = linprog(*args, method="highs", options=OPCIONES_HIGHS, **kw)
        if res.status != 0:
            raise RuntimeError(f"LP no óptimo (status {res.status}): {res.message}")
        return res

    def _matriz_cortes(self, t, k):
        clave = (t, k)
        if clave not in self._cache:
            alphas, betas = self.cortes[t][k]
            if alphas:
                B = np.vstack(betas)
                A_ub = np.hstack([B, -np.ones((len(alphas), 1))])    # beta^T x - theta <= -alpha
                self._cache[clave] = (A_ub, -np.asarray(alphas))
            else:
                self._cache[clave] = (None, None)
        return self._cache[clave]

    # ---- problemas locales ---------------------------------------------
    def resolver_nodo(self, t, k, x_padre):
        """NLDS(t,k): maestro local del nodo no terminal con la decisión del padre fija."""
        if t == 1:
            A_eq, b_eq = self._A_eq_raiz, np.array([self.b])
        else:
            T_row = self.arbol.R_rama[t][k]                         # T_{t-1,k}
            A_eq, b_eq = self._A_eq_nodo, np.array([0.0 - T_row @ x_padre])   # h - T x_padre
        A_ub, b_ub = self._matriz_cortes(t, k)
        res = self._lp(self._c_nodo, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                       bounds=self._bounds_nodo)
        return {"x": res.x[:3].copy(), "theta": float(res.x[3]), "v": float(res.fun),
                "pi": float(res.eqlin.marginals[0])}

    def resolver_hoja(self, k, x_padre):
        """LP terminal: min lam*w - y  s.a.  -y + w = M - R^T x_padre,  y, w >= 0."""
        R_row = self.arbol.R_rama[4][k]
        rhs = self.M - R_row @ x_padre                              # h - T x_padre
        res = self._lp(np.array([-1.0, self.lam]), A_eq=np.array([[-1.0, 1.0]]),
                       b_eq=np.array([rhs]), bounds=[(0, None), (0, None)])
        return {"y": float(res.x[0]), "w": float(res.x[1]), "v": float(res.fun),
                "pi": float(res.eqlin.marginals[0]), "riqueza": float(R_row @ x_padre)}

    # ---- pasadas -------------------------------------------------------
    def _forward(self):
        A = self.arbol
        X = {t: np.zeros((A.K[t], 3)) for t in (1, 2, 3)}
        TH = {t: np.zeros(A.K[t]) for t in (1, 2, 3)}
        V = {t: np.zeros(A.K[t]) for t in (1, 2, 3)}
        PI = {t: np.zeros(A.K[t]) for t in (1, 2, 3)}
        r = self.resolver_nodo(1, 0, None)
        X[1][0], TH[1][0], V[1][0], PI[1][0] = r["x"], r["theta"], r["v"], r["pi"]
        LB = r["v"]                                                  # objetivo del maestro raíz
        for t in (2, 3):
            for k in range(A.K[t]):
                r = self.resolver_nodo(t, k, X[t - 1][A.padre[t][k]])
                X[t][k], TH[t][k], V[t][k], PI[t][k] = r["x"], r["theta"], r["v"], r["pi"]
        Y, W, V4, PI4 = (np.zeros(A.K[4]) for _ in range(4))
        W4 = np.zeros(A.K[4])
        for k in range(A.K[4]):
            h = self.resolver_hoja(k, X[3][A.padre[4][k]])
            Y[k], W4[k], V4[k], PI4[k] = h["y"], h["w"], h["v"], h["pi"]
        costo = float(A.p[4] @ V4)                                   # sum p (lam w - y), con las 512 hojas
        return {"X": X, "TH": TH, "V": V, "PI": PI, "Y": Y, "W": W4, "v4": V4,
                "pi4": PI4, "LB": LB, "costo": costo}

    def _agregar_corte(self, t, k, alpha, beta, ciclo):
        self.cortes[t][k][0].append(float(alpha))
        self.cortes[t][k][1].append(np.asarray(beta, dtype=float).copy())
        self._cache.pop((t, k), None)
        if (t, k) in self.registro_cortes:
            self.registro_cortes[(t, k)].append(
                {"ciclo": ciclo, "alpha": float(alpha), "beta": np.asarray(beta).copy()})

    def _backward(self, fw, ciclo):
        """De t=3 a t=1: corte de cada padre con sus hijos, ponderado por q_{l|k}."""
        A = self.arbol
        nuevos = {t: set() for t in (1, 2, 3)}
        n_nuevos = 0
        for t in (3, 2, 1):
            for k in range(A.K[t]):
                xbar = fw["X"][t][k]
                alpha, beta = 0.0, np.zeros(3)
                for l in A.hijos(t, k):
                    if t + 1 == 4:
                        v, pi = fw["v4"][l], fw["pi4"][l]
                    elif l in nuevos[t + 1]:
                        # el hijo recibió cortes nuevos: se resuelve de nuevo en el
                        # punto heredado antes de usar sus duales
                        r = self.resolver_nodo(t + 1, l, xbar)
                        v, pi = r["v"], r["pi"]
                    else:
                        v, pi = fw["V"][t + 1][l], fw["PI"][t + 1][l]
                    beta_l = -pi * A.R_rama[t + 1][l]               # -T^T pi
                    alpha_l = v - beta_l @ xbar                     # v - beta^T xbar
                    q = self.qc[t + 1][l]
                    alpha += q * alpha_l
                    beta += q * beta_l
                theta_bar = alpha + beta @ xbar                      # valor del corte en la decisión actual
                if theta_bar > fw["TH"][t][k] + self.tol_corte * max(1.0, abs(theta_bar)):
                    self._agregar_corte(t, k, alpha, beta, ciclo)
                    nuevos[t].add(k)
                    n_nuevos += 1
        return n_nuevos

    # ---- ciclo principal -------------------------------------------------
    def resolver(self):
        t0 = time.time()
        UB, mejor, motivo = np.inf, None, "límite de ciclos"
        hist = []
        for ciclo in range(1, self.max_ciclos + 1):
            fw = self._forward()
            LB = fw["LB"]
            if fw["costo"] < UB:
                UB, mejor = fw["costo"], fw
            gap = (UB - LB) / max(1.0, abs(UB), abs(LB))
            hist.append({"ciclo": ciclo, "tiempo": time.time() - t0, "LB": LB,
                         "UB": UB, "costo_ciclo": fw["costo"], "gap": gap,
                         "cortes_total": self.n_cortes()})
            if self.verbose and (ciclo <= 5 or ciclo % 10 == 0):
                print(f"  ciclo {ciclo:4d} | LB = {LB:12.6f} | UB = {UB:12.6f} | "
                      f"gap = {gap:9.2e} | cortes = {self.n_cortes():5d}")
            if gap <= self.tol:
                motivo = "tolerancia alcanzada"
                break
            if time.time() - t0 > self.tiempo_limite:
                motivo = "límite de tiempo"
                break
            nuevos = self._backward(fw, ciclo)
            hist[-1]["cortes_nuevos"] = nuevos
            if nuevos == 0:
                motivo = "ningún corte violado"
                break
        hist = pd.DataFrame(hist)
        res = {"motivo": motivo, "ciclos": int(hist["ciclo"].iloc[-1]), "LB": float(hist["LB"].iloc[-1]),
               "UB": float(UB), "gap": float(hist["gap"].iloc[-1]),
               "cortes": self.n_cortes(), "n_lp": self.n_lp,
               "tiempo": time.time() - t0, "historial": hist, "politica": mejor,
               "L": self.L, "Rmax": self.Rmax, "registro_cortes": self.registro_cortes}
        if self.verbose:
            print(f"L-shaped anidado{' (' + self.nombre + ')' if self.nombre else ''}: {motivo} | "
                  f"ciclos = {res['ciclos']} | LB = {res['LB']:.8f} | UB = {res['UB']:.8f} | "
                  f"gap = {res['gap']:.2e} | cortes = {res['cortes']} | "
                  f"LP = {res['n_lp']} | {res['tiempo']:.1f} s")
        return res


def dif_relativa(a, b):
    """Diferencia relativa con la misma normalización que el criterio de parada."""
    return abs(a - b) / max(1.0, abs(a), abs(b))

### 5.3 Verificaciones antes de correr los experimentos

1. **Signo del dual.** Se compara $\beta=-T^\top\pi$ (con $\pi$ tomado de `marginals`)
   contra la derivada numérica de $v(\bar x)$ respecto a la decisión del padre.
2. **Problema terminal.** El LP resuelto con HiGHS debe coincidir con la forma
   cerrada $v(V)=\max\{\lambda(M-V),\,M-V\}$ y su dual con $\pi\in\{1,\lambda\}$.

In [ ]:
def verificar_signo_dual(solver, t, k, x_padre, delta=1e-4):
    """Compara beta = -pi * T con la derivada numérica de v respecto a x_padre."""
    base = solver.resolver_nodo(t, k, x_padre) if t < 4 else None
    T_row = solver.arbol.R_rama[t][k]
    beta = -base["pi"] * T_row
    num = np.zeros(3)
    for i in range(3):
        xp = x_padre.copy()
        xp[i] += delta
        num[i] = (solver.resolver_nodo(t, k, xp)["v"] - base["v"]) / delta
    return beta, num


def verificar_hoja(solver, k, x_padre):
    h = solver.resolver_hoja(k, x_padre)
    V = h["riqueza"]
    v_cerrada = max(solver.lam * (solver.M - V), solver.M - V)
    pi_cerrado = solver.lam if V < solver.M else 1.0
    return h, v_cerrada, pi_cerrado


_vs = LShapedAnidado(arbol, verbose=False)
for _V in (95.0, 110.0):            # una hoja con déficit y una con exceso
    _x = np.array([_V, 0.0, 0.0]) / arbol.R_rama[4][0][0]
    _h, _vc, _pc = verificar_hoja(_vs, 0, _x)
    print(f"hoja con V = {_h['riqueza']:.3f}: v_LP = {_h['v']:.6f}, v_cerrada = {_vc:.6f}, "
          f"pi_LP = {_h['pi']:.3f}, pi_cerrado = {_pc:.3f}")
    assert abs(_h["v"] - _vc) < 1e-8 and abs(_h["pi"] - _pc) < 1e-8
print("Problema terminal verificado contra su forma cerrada y su dual.")

## 6. Instancia reducida: L-shaped anidado frente al equivalente determinista

Se eligen **dos estados conjuntos presentes en los datos**: los dos con más
observaciones en la calibración. Se conservan sus factores $\hat R(u)$, se
renormalizan sus probabilidades (cada una dividida por la suma de ambas) y se
mantienen los tres períodos. El árbol resultante tiene 15 nodos, 37 variables y
15 igualdades.

In [ ]:
orden_frec = sorted(range(len(ESTADOS)), key=lambda s: (-Q_ESTADO[s], ESTADOS[s]))[:2]
orden_frec = sorted(orden_frec)
est_red = [ESTADOS[s] for s in orden_frec]
q_red = Q_ESTADO[orden_frec] / Q_ESTADO[orden_frec].sum()
arbol_red = Arbol(est_red, q_red, R_ESTADO[orden_frec])
print("Estados elegidos:", est_red, "| probabilidades renormalizadas:", np.round(q_red, 6))
print("Nodos del árbol reducido:", sum(arbol_red.K.values()))

de_red = equivalente_determinista(arbol_red)
print(f"Equivalente determinista reducido: {de_red['n_vars']} variables, "
      f"{de_red['n_eq']} igualdades, objetivo = {de_red['obj']:.10f}")
assert (de_red["n_vars"], de_red["n_eq"]) == (37, 15)

solver_red = LShapedAnidado(arbol_red, nombre="reducida")
ls_red = solver_red.resolver()
dif_red = dif_relativa(ls_red["UB"], de_red["obj"])
print(f"Diferencia relativa L-shaped vs equivalente determinista: {dif_red:.3e} "
      f"({'COINCIDEN' if dif_red <= TOL_LSHAPED else 'NO COINCIDEN'} a {TOL_LSHAPED:g})")
assert dif_red <= max(TOL_LSHAPED, 1e-6)

# Verificación del signo del dual en un nodo con cortes: beta = -pi*T debe coincidir con
# la derivada numérica de v(x_padre) en un punto genérico (fuera de los quiebres).
_xg = np.array([37.3, 41.1, 21.6])
_beta, _num = verificar_signo_dual(solver_red, 3, 0, _xg)
print("beta = -pi*T (marginals):", np.round(_beta, 6))
print("derivada numérica de v  :", np.round(_num, 6))
assert np.allclose(_beta, _num, atol=1e-5), "El signo del dual no coincide con la derivada numérica"
print("Signo del dual verificado: beta = -T' pi con pi = res.eqlin.marginals.")

## 7. Instancia completa (8 estados, 585 nodos): L-shaped anidado

Se registran, en cada ciclo, $LB$ (objetivo del maestro raíz), $UB$ (menor costo
esperado de las políticas completas, calculado con las 512 hojas y sus
probabilidades, no con los valores de $\theta$), la brecha y el número acumulado de
cortes. Parada: brecha $\le10^{-6}$.

In [ ]:
solver_full = LShapedAnidado(arbol, nombre="completa")
ls = solver_full.resolver()
RP = ls["UB"]                      # costo óptimo estocástico (RP)
hist = ls["historial"]
hist.to_csv(os.path.join(DIR_RES, "convergencia_lshaped.csv"), index=False)

assert ls["LB"] <= ls["UB"] + 1e-7, "LB > UB: revisar cortes y evaluación"
assert (np.diff(hist["LB"].to_numpy()) >= -1e-7).all(), "LB debe ser no decreciente"
print(f"\nLB = {ls['LB']:.8f} | UB = {ls['UB']:.8f} | brecha = {ls['gap']:.3e}")
print(f"Ciclos = {ls['ciclos']} | cortes = {ls['cortes']} | LP resueltos = {ls['n_lp']} | "
      f"tiempo = {ls['tiempo']:.1f} s | motivo: {ls['motivo']}")
print(f"Cota inferior inicial L = {ls['L']:.4f} (Rmax = {ls['Rmax']:.5f})")

# Comparación adicional con el equivalente determinista completo (no sustituye al algoritmo)
dif_de_full = dif_relativa(ls["UB"], de_full_tamano["obj"])
print(f"Equivalente determinista completo: {de_full_tamano['obj']:.8f} "
      f"({de_full_tamano['tiempo']:.2f} s) | diferencia relativa con el UB: {dif_de_full:.3e}")

In [ ]:
def figura_convergencia(h):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    axes[0].plot(h["ciclo"], h["LB"], label="LB", color="#1f77b4")
    axes[0].plot(h["ciclo"], h["UB"], label="UB", color="#d62728")
    axes[0].set_title("Cotas por ciclo", fontsize=10)
    axes[0].set_xlabel("ciclo")
    axes[0].set_ylabel("costo esperado (miles de USD)")
    axes[0].legend(fontsize=8)
    pos = h["gap"] > 0
    axes[1].semilogy(h.loc[pos, "ciclo"], h.loc[pos, "gap"], color="#2ca02c")
    axes[1].axhline(TOL_LSHAPED, color="gray", ls=":", label=f"tolerancia {TOL_LSHAPED:g}")
    axes[1].set_title("Brecha relativa", fontsize=10)
    axes[1].set_xlabel("ciclo")
    axes[1].legend(fontsize=8)
    axes[2].plot(h["ciclo"], h["cortes_total"], color="#9467bd")
    axes[2].set_title("Cortes acumulados", fontsize=10)
    axes[2].set_xlabel("ciclo")
    fig.tight_layout()
    return fig


fig = figura_convergencia(hist)
fig.savefig(os.path.join(DIR_FIG, "fig_convergencia.png"), dpi=200, bbox_inches="tight")
plt.show()

### 7.1 Evidencia: un corte numérico en etapa 3, otro en la raíz, y el LP terminal con su dual

In [ ]:
def tabla_corte(rc, etiqueta):
    return {"nodo": etiqueta, "ciclo": rc["ciclo"], "alpha": rc["alpha"],
            "b_SPY": rc["beta"][0], "b_LQD": rc["beta"][1], "b_GLD": rc["beta"][2]}


reg = ls["registro_cortes"]
cortes_ev = [tabla_corte(reg[(3, 0)][0], "(3,1) primer corte"),
             tabla_corte(reg[(3, 0)][-1], "(3,1) último corte"),
             tabla_corte(reg[(1, 0)][0], "(1,1) primer corte"),
             tabla_corte(reg[(1, 0)][-1], "(1,1) último corte")]
tab_cortes = pd.DataFrame(cortes_ev)
print("Cortes de la forma  theta >= alpha + beta' x  (x en miles de USD)")
display(tab_cortes.round(6))

# LP terminal y su dual en una hoja de la política óptima (con déficit si existe)
pol = ls["politica"]
X = pol["X"]
V4 = np.einsum("ki,ki->k", arbol.R_rama[4], X[3][arbol.padre[4]])
k_ej = int(np.argmax(pol["W"])) if pol["W"].max() > 1e-9 else 0
h_ej = solver_full.resolver_hoja(k_ej, X[3][arbol.padre[4][k_ej]])
print(f"\nHoja ejemplo k = {k_ej + 1}, historia {arbol.historia(4, k_ej)}, "
      f"p = {arbol.p[4][k_ej]:.6f}")
print(f"  LP primal: min {LAMBDA:g} w - y   s.a.  -y + w = M - V = {META - h_ej['riqueza']:.6f},  y,w >= 0")
print(f"  solución primal: y = {h_ej['y']:.6f}, w = {h_ej['w']:.6f}, valor = {h_ej['v']:.6f}")
print(f"  LP dual : max pi (M - V)  s.a.  1 <= pi <= {LAMBDA:g}   ->  pi = {h_ej['pi']:.6f}, "
      f"valor dual = {h_ej['pi'] * (META - h_ej['riqueza']):.6f}")
assert abs(h_ej["pi"] * (META - h_ej["riqueza"]) - h_ej["v"]) < 1e-8      # dualidad fuerte

## 8. Política obtenida y comparación dentro del árbol

In [ ]:
Vt = {t: X[t].sum(axis=1) for t in (1, 2, 3)}                  # riqueza de cada nodo bajo la política
F = {t: X[t] / Vt[t][:, None] for t in (1, 2, 3)}              # proporciones f^t_{i,k}
assert all(np.allclose(F[t].sum(axis=1), 1.0) and F[t].min() >= -1e-9 for t in (1, 2, 3))
assert abs(Vt[1][0] - B0) < 1e-8

filas = []
for t in (1, 2, 3):
    for k in range(arbol.K[t]):
        filas.append({"etapa": t, "k": k + 1, "historia": arbol.historia(t, k),
                      "riqueza": Vt[t][k],
                      **{f"x_{a}": X[t][k, i] for i, a in enumerate(ACTIVOS)},
                      **{f"f_{a}": F[t][k, i] for i, a in enumerate(ACTIVOS)}})
tab_politica = pd.DataFrame(filas)
tab_politica.to_csv(os.path.join(DIR_RES, "politica_73_nodos.csv"), index=False)
print("Raíz y nodos de etapa 2 (primeras 9 de las 73 filas):")
display(tab_politica.head(9).round(5))

In [ ]:
def figura_politica(F, arbol):
    etiquetas = ["Raíz"] + [arbol.historia(2, k) for k in range(arbol.K[2])]
    props = np.vstack([F[1], F[2]])
    fig, ax = plt.subplots(figsize=(8, 3.8))
    base = np.zeros(len(etiquetas))
    for i, a in enumerate(ACTIVOS):
        ax.bar(etiquetas, props[:, i], bottom=base, label=a)
        base += props[:, i]
    ax.set_ylabel("proporción de la riqueza")
    ax.set_title("Política óptima: proporciones en la raíz y en los 8 nodos de la etapa 2")
    ax.set_ylim(0, 1)
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8)
    fig.tight_layout()
    return fig


fig = figura_politica(F, arbol)
fig.savefig(os.path.join(DIR_FIG, "fig_politica_etapa2.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Hojas: probabilidad, riqueza, exceso, déficit y costo (512 filas)
Y_h, W_h = pol["Y"], pol["W"]
assert np.allclose(Y_h, np.maximum(V4 - META, 0), atol=1e-7)   # LP terminal = forma cerrada
assert np.allclose(W_h, np.maximum(META - V4, 0), atol=1e-7)
costo_h = LAMBDA * W_h - Y_h
tab_hojas = pd.DataFrame({"k": np.arange(1, arbol.K[4] + 1),
                          "historia": [arbol.historia(4, k) for k in range(arbol.K[4])],
                          "prob": arbol.p[4], "riqueza": V4, "exceso": Y_h,
                          "deficit": W_h, "costo": costo_h})
tab_hojas.to_csv(os.path.join(DIR_RES, "hojas_512.csv", ), index=False)
assert abs(arbol.p[4] @ costo_h - RP) < 1e-7


def cuantil_ponderado(x, p, alfa):
    o = np.argsort(x)
    c = np.cumsum(p[o])
    return float(x[o][np.searchsorted(c, alfa)])


def agregados(V4, p):
    y, w = np.maximum(V4 - META, 0), np.maximum(META - V4, 0)
    media = float(p @ V4)
    return {"costo_esperado": float(p @ (LAMBDA * w - y)), "riqueza_esperada": media,
            "riqueza_desv": float(np.sqrt(p @ (V4 - media) ** 2)),
            "riqueza_p05": cuantil_ponderado(V4, p, 0.05),
            "deficit_esperado": float(p @ w), "exceso_esperado": float(p @ y),
            "prob_meta": float(p[V4 >= META - TOL_META].sum()),
            "prob_deficit": float(p[w > TOL_META].sum()),
            "riqueza_min": float(V4.min()), "riqueza_max": float(V4.max())}


agg_sto = agregados(V4, arbol.p[4])
print("Política estocástica (RP):")
display(pd.Series(agg_sto).round(5).to_frame("valor").T)

### 8.1 Modelo de valor esperado (EV), EEV y VSS

El modelo EV reemplaza, en cada activo y período, el factor por
$\bar R_i=\sum_u\hat q_u\hat R_i(u)$ y optimiza un único camino. Sus proporciones
$g^t_i$ se aplican **sin reoptimizar** a las 512 trayectorias del árbol (EEV).
$VSS=EEV-RP$ (problema de minimización).

In [ ]:
def modelo_EV(arbol, b=B0, M=META, lam=LAMBDA):
    """LP de un solo camino con los factores promedio R_bar_i = sum_u q_u R_i(u)."""
    Rbar = arbol.q @ arbol.Rhat
    c = np.r_[np.zeros(9), -1.0, lam]                         # x1(3), x2(3), x3(3), y, w
    A = np.zeros((4, 11))
    A[0, 0:3] = 1.0                                           # sum x1 = b
    A[1, 0:3], A[1, 3:6] = Rbar, -1.0                         # Rbar x1 - sum x2 = 0
    A[2, 3:6], A[2, 6:9] = Rbar, -1.0                         # Rbar x2 - sum x3 = 0
    A[3, 6:9], A[3, 9], A[3, 10] = Rbar, -1.0, 1.0            # Rbar x3 - y + w = M
    res = linprog(c, A_eq=A, b_eq=np.array([b, 0.0, 0.0, M]), bounds=(0, None),
                  method="highs", options=OPCIONES_HIGHS)
    assert res.status == 0, res.message
    x = {t: res.x[3 * (t - 1):3 * t] for t in (1, 2, 3)}
    g = np.vstack([x[t] / x[t].sum() for t in (1, 2, 3)])    # g^t_i = x^t_i / V^t
    return {"Rbar": Rbar, "x": x, "g": g, "obj": float(res.fun), "n_vars": 11, "n_eq": 4,
            "V": {t: float(x[t].sum()) for t in (1, 2, 3)}}


ev = modelo_EV(arbol)
G = ev["g"]
print("Factores promedio R_bar:", dict(zip(ACTIVOS, np.round(ev["Rbar"], 6))))
print("Objetivo del modelo EV (no comparable con RP):", round(ev["obj"], 6))
display(pd.DataFrame(G, index=["g1 (dic-25)", "g2 (mar-26)", "g3 (jun-26)"], columns=ACTIVOS).round(6))


def riqueza_regla_en_hojas(arbol, g, b=B0):
    """Aplica proporciones fijas por etapa a las 512 trayectorias y devuelve la riqueza terminal."""
    S = arbol.S
    k = np.arange(arbol.K[4])
    z1, z2, z3 = k // S ** 2, (k // S) % S, k % S
    V2 = b * (arbol.Rhat[z1] @ g[0])
    V3 = V2 * (arbol.Rhat[z2] @ g[1])
    return V3 * (arbol.Rhat[z3] @ g[2])


V4_ev = riqueza_regla_en_hojas(arbol, G)
agg_ev = agregados(V4_ev, arbol.p[4])
EEV = agg_ev["costo_esperado"]
VSS = EEV - RP
# la política estocástica expresada como proporciones por nodo reproduce las hojas del LP
_S = arbol.S
_k = np.arange(arbol.K[4]); _z1, _z2, _z3 = _k // _S ** 2, (_k // _S) % _S, _k % _S
_V2 = B0 * (arbol.Rhat[_z1] @ F[1][0])
_V3 = _V2 * 0 + np.einsum("ki,ki->k", arbol.Rhat[_z2], F[2][_z1] * _V2[:, None])
_V4 = np.einsum("ki,ki->k", arbol.Rhat[_z3], F[3][_z1 * _S + _z2] * _V3[:, None])
assert np.allclose(_V4, V4, atol=1e-7), "las proporciones por nodo no reproducen la política"

print(f"RP  (costo óptimo estocástico) = {RP:.6f}")
print(f"EEV (costo esperado de la regla EV) = {EEV:.6f}")
print(f"VSS = EEV - RP = {VSS:.6f}")
assert VSS >= -1e-6, "VSS negativo: error numérico o de implementación"

tab_comp = pd.DataFrame({"Estocástica": agg_sto, "Regla EV": agg_ev}).T
display(tab_comp.round(5))

In [ ]:
def figura_riqueza_terminal(V4_s, V4_e, p):
    fig, ax = plt.subplots(figsize=(8, 3.8))
    lo, hi = min(V4_s.min(), V4_e.min()), max(V4_s.max(), V4_e.max())
    bins = np.linspace(lo, hi, 41)
    ax.hist(V4_s, bins=bins, weights=p, alpha=0.65, label="Estocástica", color="#1f77b4")
    ax.hist(V4_e, bins=bins, weights=p, alpha=0.55, label="Regla EV", color="#ff7f0e")
    ax.axvline(META, color="black", ls="--", lw=1, label=f"meta M = {META:g}")
    ax.set_xlabel("riqueza terminal (miles de USD)")
    ax.set_ylabel("probabilidad")
    ax.set_title("Distribución de la riqueza terminal ponderada por probabilidades (512 hojas)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


fig = figura_riqueza_terminal(V4, V4_ev, arbol.p[4])
fig.savefig(os.path.join(DIR_FIG, "fig_riqueza_terminal.png"), dpi=200, bbox_inches="tight")
plt.show()

## 9. Validación fuera de muestra: enero–septiembre de 2026

**Regla de aplicación (fijada antes de mirar 2026).** La terna de signos de los
rendimientos observados determina el nodo; las proporciones guardadas de ese nodo
se aplican a la riqueza **realmente disponible**. Los factores reales (no los
promedios del estado) actualizan la riqueza. Si se observara un estado sin soporte
(rama eliminada en la calibración), se reparte la riqueza en partes iguales en los
períodos restantes.

In [ ]:
P_val = val[ACTIVOS].to_numpy(dtype=float)                # 4 cierres: dic-25, mar, jun, sep-26
R_real = P_val[1:] / P_val[:-1]                           # factores reales (3, 3)
z_real = [terna(R_real[t] - 1.0) for t in range(3)]
idx_estado = {u: i for i, u in enumerate(ESTADOS)}
fechas_val = [d.strftime("%d-%b-%Y") for d in val["fecha_cierre"]]
print("Factores reales:")
display(pd.DataFrame(R_real, index=["T1 (mar-26)", "T2 (jun-26)", "T3 (sep-26)"], columns=ACTIVOS).round(5))
print("Estados observados:", z_real, "| con soporte en el árbol:", [z in idx_estado for z in z_real])


def aplicar_regla(prop, usa_nodos):
    """Aplica una regla de proporciones a la trayectoria real; cada política actualiza su riqueza."""
    V, k, respaldo, filas = B0, 0, False, []
    for tau in (1, 2, 3):
        if respaldo:
            f, nodo = np.full(3, 1.0 / 3.0), "respaldo"
        else:
            f, nodo = prop(tau, k), (f"({tau},{k + 1})" if usa_nodos else "—")
        x = f * V
        assert x.min() >= -1e-9 and abs(x.sum() - V) < 1e-9          # no negatividad y suma = riqueza
        V_fin = float(R_real[tau - 1] @ x)
        filas.append({"tau": tau, "trimestre": ["T1 (ene–mar)", "T2 (abr–jun)", "T3 (jul–sep)"][tau - 1],
                      "nodo": nodo, "V_ini": V, **{f"f_{a}": f[i] for i, a in enumerate(ACTIVOS)},
                      **{f"x_{a}": x[i] for i, a in enumerate(ACTIVOS)},
                      **{f"R_{a}": R_real[tau - 1, i] for i, a in enumerate(ACTIVOS)}, "V_fin": V_fin})
        if usa_nodos and tau < 3 and not respaldo:
            z = z_real[tau - 1]
            if z in idx_estado:
                k = k * arbol.S + idx_estado[z]          # nodo de la historia ya conocida
            else:
                respaldo = True
        V = V_fin
    return pd.DataFrame(filas), respaldo


val_sto, respaldo_sto = aplicar_regla(lambda tau, k: F[tau][k], True)
val_ev, _ = aplicar_regla(lambda tau, k: G[tau - 1], False)
print("¿Se activó el respaldo de estado sin soporte?", respaldo_sto)
print("\nPolítica estocástica")
display(val_sto.round(5))
print("\nRegla EV")
display(val_ev.round(5))


def resultado_final(df):
    V4r = float(df["V_fin"].iloc[-1])
    exc, dfc = max(V4r - META, 0.0), max(META - V4r, 0.0)
    return {"riqueza_terminal": V4r, "rend_acumulado": V4r / B0 - 1.0, "exceso": exc,
            "deficit": dfc, "costo_realizado": LAMBDA * dfc - exc,
            "cumple_meta": int(V4r >= META - TOL_META)}


fin_sto, fin_ev = resultado_final(val_sto), resultado_final(val_ev)
tab_final = pd.DataFrame({"Estocástica": fin_sto, "Regla EV": fin_ev}).T
display(tab_final.round(5))
val_sto.to_csv(os.path.join(DIR_RES, "validacion_estocastica.csv"), index=False)
val_ev.to_csv(os.path.join(DIR_RES, "validacion_ev.csv"), index=False)

In [ ]:
def figura_validacion(val_sto, val_ev):
    ejes_x = ["dic-2025", "mar-2026", "jun-2026", "sep-2026"]
    w_s = [B0] + val_sto["V_fin"].tolist()
    w_e = [B0] + val_ev["V_fin"].tolist()
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    ax.plot(ejes_x, w_s, "o-", label="Estocástica", color="#1f77b4")
    ax.plot(ejes_x, w_e, "s--", label="Regla EV", color="#ff7f0e")
    ax.axhline(META, color="black", ls=":", lw=1, label=f"meta M = {META:g}")
    for xs, a, b_ in zip(ejes_x, w_s, w_e):
        ax.annotate(f"{a:.2f}", (xs, a), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7)
    ax.set_ylabel("riqueza (miles de USD)")
    ax.set_title("Riqueza realizada de cada política, enero–septiembre de 2026")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


fig = figura_validacion(val_sto, val_ev)
fig.savefig(os.path.join(DIR_FIG, "fig_validacion.png"), dpi=200, bbox_inches="tight")
plt.show()

## 10. Diagnósticos para las conclusiones

Cálculos que respaldan las preguntas de la sección 5.4 del enunciado: cambio de
composición tras cada observación, diferencia entre la riqueza del árbol y la
observada, activos sin inversión, dominancia entre activos y variabilidad que se
pierde al resumir cada grupo con su media.

In [ ]:
S_ = len(ESTADOS)
nombres_tri = ["dic-2025", "mar-2026", "jun-2026"]


def pct(x, d=1):
    return f"{100 * x:.{d}f}\\%"


# (a) Dominancia: ¿algún activo tiene factor medio >= al de otro en todos los estados?
dom_cuentas = {}
for i, j in itertools.permutations(range(3), 2):
    dom_cuentas[(ACTIVOS[i], ACTIVOS[j])] = int((R_ESTADO[:, i] >= R_ESTADO[:, j] - 1e-15).sum())
dominancias = [(a, b) for (a, b), n in dom_cuentas.items() if n == S_]
print("Pares con dominancia en los", S_, "estados:", dominancias if dominancias else "ninguno")
for (a, b), n in dom_cuentas.items():
    print(f"  R_{a}(u) >= R_{b}(u) en {n}/{S_} estados")

# (b) Activos sin inversión
umbral = 1e-6
sin_inv_nodos = {a: int(sum((F[t][:, i] < umbral).sum() for t in (1, 2, 3)))
                 for i, a in enumerate(ACTIVOS)}
sin_inv_raiz = [a for i, a in enumerate(ACTIVOS) if F[1][0, i] < umbral]
sin_inv_traye = {t + 1: [a for i, a in enumerate(ACTIVOS) if val_sto["f_" + a].iloc[t] < umbral]
                 for t in range(3)}
print("Nodos (de 73) con 0% en cada activo:", sin_inv_nodos)
print("Activos sin inversión en la raíz:", sin_inv_raiz or "ninguno")
print("Activos sin inversión en la trayectoria validada:", sin_inv_traye)

# (c) Cambio de composición tras cada observación (política estocástica en 2026)
comp = [val_sto[[f"f_{a}" for a in ACTIVOS]].iloc[t].to_numpy() for t in range(3)]
for t in range(3):
    print(f"{nombres_tri[t]}: " + ", ".join(f"{a} {100 * comp[t][i]:.1f}%" for i, a in enumerate(ACTIVOS))
          + f" (nodo {val_sto['nodo'].iloc[t]})")

# (d) Riqueza del árbol frente a riqueza observada, y por qué difieren
k_nodos = [0]
for t in (0, 1):
    z = z_real[t]
    k_nodos.append(k_nodos[-1] * arbol.S + idx_estado[z]
                   if (k_nodos[-1] is not None and z in idx_estado) else None)
dif_riqueza = []
for t in range(3):
    if k_nodos[t] is None:
        continue
    v_arbol = float(Vt[t + 1][k_nodos[t]])
    v_real = float(val_sto["V_ini"].iloc[t])
    dif_riqueza.append({"inicio": nombres_tri[t], "nodo": f"({t + 1},{k_nodos[t] + 1})",
                        "V_arbol": v_arbol, "V_real": v_real, "dif": v_real - v_arbol})
tab_riq = pd.DataFrame(dif_riqueza)
display(tab_riq.round(4))
dif_factores = []
for t in range(2):
    z = z_real[t]
    if z in idx_estado:
        Rh = R_ESTADO[idx_estado[z]]
        dif_factores.append({"trimestre": t + 1, "estado": z,
                             **{f"R_real_{a}": R_real[t, i] for i, a in enumerate(ACTIVOS)},
                             **{f"R_hat_{a}": Rh[i] for i, a in enumerate(ACTIVOS)}})
display(pd.DataFrame(dif_factores).round(5))

# (e) Variabilidad: el árbol resume cada grupo con su media
media_ar = arbol.q @ arbol.Rhat
sd_arbol = np.sqrt(arbol.q @ (arbol.Rhat - media_ar) ** 2)
sd_hist = r_hist.std(axis=0, ddof=0)           # misma convención (población) que el árbol
tab_var = pd.DataFrame({"sd_hist_%": 100 * sd_hist, "sd_arbol_%": 100 * sd_arbol,
                        "cociente": sd_arbol / sd_hist,
                        "min_hist_%": 100 * r_hist.min(axis=0),
                        "min_arbol_%": 100 * (arbol.Rhat.min(axis=0) - 1.0)}, index=ACTIVOS)
display(tab_var.round(3))

dentro = []
for u in ESTADOS:
    idx = [j for j, z in enumerate(z_hist) if z == u]
    dentro.append({"estado": u, "n": len(idx),
                   **{f"sd_{a}_%": 100 * (R_hist[idx, i].std(ddof=1) if len(idx) > 1 else np.nan)
                      for i, a in enumerate(ACTIVOS)}})
print("Desviación estándar de R dentro de cada grupo (en %):")
display(pd.DataFrame(dentro).round(3))

## 11. Exportar tablas y macros para el informe

Todas las cifras del informe salen de aquí: se escriben archivos `.tex` en
`resultados/` que el informe incluye con `\input`. Así el PDF refleja exactamente
la ejecución de este notebook.

In [ ]:
def tex_tabla(nombre, encabezados, filas, alin=None, tam=None):
    alin = alin or "l" + "r" * (len(encabezados) - 1)
    L = ["{\\" + tam] if tam else []
    L += [f"\\begin{{tabular}}{{{alin}}}", "\\toprule", " & ".join(encabezados) + " \\\\", "\\midrule"]
    L += [" & ".join(str(c) for c in f) + " \\\\" for f in filas]
    L += ["\\bottomrule", "\\end{tabular}"] + (["}"] if tam else [])
    with open(os.path.join(DIR_RES, nombre), "w", encoding="utf-8") as fh:
        fh.write("\n".join(L) + "\n")


def f_(x, d=4):
    return f"{x:.{d}f}"


# --- datos y árbol ---
tex_tabla("tab_estadisticas.tex",
          ["Activo", "Media (\\%)", "Desv. est. (\\%)", "Mín. (\\%)", "Máx. (\\%)"],
          [[a] + [f_(100 * est.loc[a, c], 2) for c in ("media", "desv_est", "min", "max")] for a in ACTIVOS])
tex_tabla("tab_estados.tex",
          ["Estado $u$", "$n_u$", "$\\hat q_u$", "$\\hat R_{\\mathrm{SPY}}(u)$",
           "$\\hat R_{\\mathrm{LQD}}(u)$", "$\\hat R_{\\mathrm{GLD}}(u)$"],
          [[r.estado, int(r.n_u), f_(r.q_u, 4)] + [f_(getattr(r, f"R_{a}"), 5) for a in ACTIVOS]
           for r in tabla_estados.itertuples()], tam="small")
tex_tabla("tab_ramas21.tex",
          ["Nodo $(3,k)$", "Historia", "Estado de la rama", "$q_{\\ell|k}$", "$p^3_\\ell$"],
          [[f"(3,{l + 1})", arbol.historia(3, l), arbol.estados[arbol.rama[3][l]],
            f_(arbol.q_cond(3, l), 4), f_(arbol.p[3][l], 5)]
           for l in arbol.hijos(2, 0)], tam="small")

# --- L-shaped ---
tex_tabla("tab_cortes.tex",
          ["Nodo", "Ciclo", "$\\alpha$", "$\\beta_{\\mathrm{SPY}}$", "$\\beta_{\\mathrm{LQD}}$", "$\\beta_{\\mathrm{GLD}}$"],
          [[r.nodo, r.ciclo, f_(r.alpha, 4), f_(r.b_SPY, 6), f_(r.b_LQD, 6), f_(r.b_GLD, 6)]
           for r in tab_cortes.itertuples()], tam="small")
tex_tabla("tab_hoja.tex",
          ["Hoja $k$", "Historia", "$V^4_k$", "$M-V^4_k$", "$y^4_k$", "$w^4_k$", "$v$", "$\\pi$"],
          [[k_ej + 1, arbol.historia(4, k_ej), f_(h_ej["riqueza"], 4), f_(META - h_ej["riqueza"], 4),
            f_(h_ej["y"], 4), f_(h_ej["w"], 4), f_(h_ej["v"], 4), f_(h_ej["pi"], 2)]], tam="small")
tex_tabla("tab_reducido.tex",
          ["Método", "Objetivo", "Variables", "Igualdades", "Ciclos", "Cortes", "Tiempo [s]"],
          [["L-shaped anidado", f_(ls_red["UB"], 8), "—", "—", ls_red["ciclos"], ls_red["cortes"], f_(ls_red["tiempo"], 2)],
           ["Equivalente determinista", f_(de_red["obj"], 8), de_red["n_vars"], de_red["n_eq"], "—", "—", f_(de_red["tiempo"], 3)]],
          alin="lrrrrrr", tam="small")
tex_tabla("tab_lshaped_full.tex",
          ["Enfoque", "Objetivo / $UB$", "$LB$", "Brecha", "Ciclos", "Cortes", "LP", "Tiempo [s]"],
          [["L-shaped anidado", f_(ls["UB"], 8), f_(ls["LB"], 8), f"{ls['gap']:.2e}", ls["ciclos"], ls["cortes"], ls["n_lp"], f_(ls["tiempo"], 1)],
           [f"Equiv. determinista ({de_full_tamano['n_vars']} var., {de_full_tamano['n_eq']} ig.)",
            f_(de_full_tamano["obj"], 8), "—", "—", "—", "—", "1", f_(de_full_tamano["tiempo"], 2)]],
          alin="lrrrrrrr", tam="small")

# --- política y comparación en el árbol ---
filas_pol = []
for t, k, nombre in [(1, 0, "(1,1)")] + [(2, k, f"(2,{k + 1})") for k in range(arbol.K[2])]:
    filas_pol.append([nombre, arbol.historia(t, k), f_(Vt[t][k], 3)]
                     + [f_(X[t][k, i], 3) for i in range(3)] + [f_(100 * F[t][k, i], 2) for i in range(3)])
tex_tabla("tab_politica.tex",
          ["Nodo", "Historia", "$V^{t,*}_k$", "$x_{\\mathrm{SPY}}$", "$x_{\\mathrm{LQD}}$", "$x_{\\mathrm{GLD}}$",
           "$f_{\\mathrm{SPY}}$ (\\%)", "$f_{\\mathrm{LQD}}$ (\\%)", "$f_{\\mathrm{GLD}}$ (\\%)"],
          filas_pol, alin="llrrrrrrr", tam="small")

metricas = [("Costo esperado", "costo_esperado", 4), ("Riqueza esperada", "riqueza_esperada", 3),
            ("Desv. est. de la riqueza", "riqueza_desv", 3), ("Percentil 5 de la riqueza", "riqueza_p05", 3),
            ("Déficit esperado", "deficit_esperado", 4), ("Exceso esperado", "exceso_esperado", 4),
            ("$P(V^4\\ge M)$", "prob_meta", 4), ("$P(\\text{déficit}>0)$", "prob_deficit", 4)]
filas_cmp = [[nom, f_(agg_sto[c], d), f_(agg_ev[c], d)] for nom, c, d in metricas]
filas_cmp.append(["$VSS=EEV-RP$", "\\multicolumn{2}{c}{" + f_(VSS, 4) + "}"])
tex_tabla("tab_comparacion.tex", ["Indicador", "Estocástica (RP)", "Regla EV (EEV)"], filas_cmp, tam="small")
tex_tabla("tab_ev.tex", ["Etapa", "$g_{\\mathrm{SPY}}$ (\\%)", "$g_{\\mathrm{LQD}}$ (\\%)", "$g_{\\mathrm{GLD}}$ (\\%)"],
          [[f"$t={t}$"] + [f_(100 * G[t - 1, i], 2) for i in range(3)] for t in (1, 2, 3)])

# --- validación ---
def tex_val(nombre, df, usa_nodos):
    filas_v = []
    for r in df.itertuples():
        filas_v.append([r.trimestre, r.nodo if usa_nodos else "—", f_(r.V_ini, 3)]
                       + [f_(100 * getattr(r, f"f_{a}"), 1) for a in ACTIVOS]
                       + [f_(getattr(r, f"x_{a}"), 3) for a in ACTIVOS] + [f_(r.V_fin, 3)])
    tex_tabla(nombre, ["Trimestre", "Nodo", "$\\tilde V$ ini.", "$f_{\\mathrm{SPY}}$ (\\%)", "$f_{\\mathrm{LQD}}$ (\\%)",
                       "$f_{\\mathrm{GLD}}$ (\\%)", "$\\tilde x_{\\mathrm{SPY}}$", "$\\tilde x_{\\mathrm{LQD}}$",
                       "$\\tilde x_{\\mathrm{GLD}}$", "$\\tilde V$ fin"],
              filas_v, alin="llrrrrrrrr", tam="footnotesize")


tex_val("tab_val_sto.tex", val_sto, True)
tex_val("tab_val_ev.tex", val_ev, False)
tex_tabla("tab_factores_reales.tex",
          ["Trimestre", "Fecha de cierre", "$\\tilde R_{\\mathrm{SPY}}$", "$\\tilde R_{\\mathrm{LQD}}$",
           "$\\tilde R_{\\mathrm{GLD}}$", "Estado $\\tilde z_\\tau$"],
          [[["Ene–mar 2026", "Abr–jun 2026", "Jul–sep 2026"][t], val["fecha_cierre"].iloc[t + 1].strftime("%Y-%m-%d")]
           + [f_(R_real[t, i], 5) for i in range(3)] + [z_real[t]] for t in range(3)], alin="llrrrr", tam="small")
tex_tabla("tab_final.tex",
          ["Política", "$\\tilde V^4$", "Rend. acum.", "Exceso", "Déficit", "Costo realizado", "Meta (1/0)"],
          [[n, f_(d["riqueza_terminal"], 4), pct(d["rend_acumulado"], 2), f_(d["exceso"], 4),
            f_(d["deficit"], 4), f_(d["costo_realizado"], 4), d["cumple_meta"]]
           for n, d in (("Estocástica", fin_sto), ("Regla EV", fin_ev))], alin="lrrrrrr", tam="small")
tex_tabla("tab_riqueza_arbol_real.tex",
          ["Inicio", "Nodo", "$V^{t,*}_k$ (árbol)", "$\\tilde V^t$ (real)", "Diferencia"],
          [[r.inicio, r.nodo, f_(r.V_arbol, 4), f_(r.V_real, 4), f_(r.dif, 4)] for r in tab_riq.itertuples()], tam="small")
tex_tabla("tab_variabilidad.tex",
          ["Activo", "Desv. hist. (\\%)", "Desv. árbol (\\%)", "Cociente", "Mín. hist. (\\%)", "Mín. árbol (\\%)"],
          [[a, f_(100 * sd_hist[i], 2), f_(100 * sd_arbol[i], 2), f_(sd_arbol[i] / sd_hist[i], 2),
            f_(100 * r_hist[:, i].min(), 2), f_(100 * (arbol.Rhat[:, i].min() - 1), 2)] for i, a in enumerate(ACTIVOS)],
          tam="small")

In [ ]:
# --- frases generadas a partir de los resultados (para la discusión del informe) ---
pares_u = [(i, j) for i in range(3) for j in range(i + 1, 3)]
if dominancias:
    txt_dom = "; ".join(f"{a} tiene un factor medio mayor o igual que {b} en los {S_} estados" for a, b in dominancias)
else:
    txt_dom = ("ningún activo tiene un factor medio mayor o igual que otro en todos los estados ("
               + "; ".join(f"{ACTIVOS[i]} $\\ge$ {ACTIVOS[j]} en {dom_cuentas[(ACTIVOS[i], ACTIVOS[j])]} de {S_} estados"
                           for i, j in pares_u) + ")")
txt_sin = (f"en la raíz " + ("recibe 0\\% " + ", ".join(sin_inv_raiz) if sin_inv_raiz else "los tres activos reciben inversión")
           + "; nodos de los 73 con 0\\% en cada activo: "
           + ", ".join(f"{a} {n}" for a, n in sin_inv_nodos.items())
           + "; en la trayectoria validada, sin inversión: "
           + "; ".join(f"{nombres_tri[t - 1]}: " + (", ".join(v) if v else "ninguno") for t, v in sin_inv_traye.items()))
txt_comp = "; ".join(
    f"{nombres_tri[t]}: " + ", ".join(f"{a} {100 * comp[t][i]:.1f}\\%" for i, a in enumerate(ACTIVOS))
    for t in range(3))
txt_riq = "; ".join(f"{r.inicio}: real {r.V_real:.3f} frente a {r.V_arbol:.3f} del nodo {r.nodo}" for r in tab_riq.itertuples())
txt_var = (", ".join(f"{a} {100 * sd_arbol[i] / sd_hist[i]:.0f}\\%" for i, a in enumerate(ACTIVOS)))

# --- macros ---
M_ = {}
def mac(nombre, valor):
    M_[nombre] = str(valor)

mac("ValLB", f_(ls["LB"], 6)); mac("ValUB", f_(ls["UB"], 6)); mac("ValGap", f"{ls['gap']:.2e}")
mac("NumCiclos", ls["ciclos"]); mac("NumCortes", ls["cortes"]); mac("NumLP", ls["n_lp"])
mac("TiempoLS", f_(ls["tiempo"], 1)); mac("MotivoLS", ls["motivo"])
mac("CotaL", f_(ls["L"], 4)); mac("FactorRmax", f_(ls["Rmax"], 5))
mac("ValRP", f_(RP, 6)); mac("ValEEV", f_(EEV, 6)); mac("ValVSS", f_(VSS, 6)); mac("ObjEV", f_(ev["obj"], 6))
mac("ValDEfull", f_(de_full_tamano["obj"], 6)); mac("DifDEfull", f"{dif_de_full:.2e}")
mac("NumVarsDE", de_full_tamano["n_vars"]); mac("NumEqDE", de_full_tamano["n_eq"])
mac("TiempoDEfull", f_(de_full_tamano["tiempo"], 2))
mac("EstRedA", est_red[0]); mac("EstRedB", est_red[1])
mac("QRedA", f_(q_red[0], 4)); mac("QRedB", f_(q_red[1], 4))
mac("ObjRedLS", f_(ls_red["UB"], 8)); mac("ObjRedDE", f_(de_red["obj"], 8)); mac("DifRed", f"{dif_red:.2e}")
mac("CiclosRed", ls_red["ciclos"]); mac("CortesRed", ls_red["cortes"])
for suf, ag in (("Sto", agg_sto), ("Ev", agg_ev)):
    mac("RiqEsp" + suf, f_(ag["riqueza_esperada"], 3)); mac("DefEsp" + suf, f_(ag["deficit_esperado"], 4))
    mac("PMeta" + suf, pct(ag["prob_meta"], 1)); mac("PDef" + suf, pct(ag["prob_deficit"], 1))
    mac("CostoArbol" + suf, f_(ag["costo_esperado"], 4))
for suf, d in (("Sto", fin_sto), ("Ev", fin_ev)):
    mac("RiqFin" + suf, f_(d["riqueza_terminal"], 3)); mac("Rend" + suf, pct(d["rend_acumulado"], 2))
    mac("CostoReal" + suf, f_(d["costo_realizado"], 3)); mac("Meta" + suf, "sí" if d["cumple_meta"] else "no")
    mac("ExcFin" + suf, f_(d["exceso"], 3)); mac("DefFin" + suf, f_(d["deficit"], 3))
mac("Respaldo", "sí" if respaldo_sto else "no")
mac("FechaConsulta", META_DESCARGA["fecha_consulta_utc"][:10])
_MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
          "septiembre", "octubre", "noviembre", "diciembre"]
_fc = val["fecha_cierre"].iloc[-1]
mac("FechaCierreSep", f"{_fc.day} de {_MESES[_fc.month - 1]} de {_fc.year}")
mac("NSoporte", S_)
_dc = fin_sto["costo_realizado"] - fin_ev["costo_realizado"]
if abs(_dc) < 1e-9:
    txt_valcomp = "Ambas políticas obtuvieron el mismo costo realizado."
else:
    _mej, _peor = ("la política estocástica", "la regla EV") if _dc < 0 else ("la regla EV", "la política estocástica")
    txt_valcomp = (f"En esta única trayectoria, {_mej} tuvo menor costo realizado que {_peor} "
                   f"(diferencia de {abs(_dc):.3f} miles de USD); esto describe lo ocurrido en esos nueve meses "
                   f"y no establece superioridad general.")
txt_vss = ("el VSS es estrictamente positivo" if VSS > 1e-6 else "el VSS es esencialmente cero")
mac("TxtValComp", txt_valcomp); mac("TxtVSS", txt_vss)
mac("TxtDominancia", txt_dom); mac("TxtSinInversion", txt_sin); mac("TxtComposicion", txt_comp)
mac("TxtRiquezaArbol", txt_riq); mac("TxtVariabilidad", txt_var)

with open(os.path.join(DIR_RES, "macros.tex"), "w", encoding="utf-8") as fh:
    for k_, v_ in M_.items():
        fh.write(f"\\newcommand{{\\{k_}}}{{{v_}}}\n")

resumen = {"LB": ls["LB"], "UB": ls["UB"], "gap": ls["gap"], "ciclos": ls["ciclos"],
           "cortes": ls["cortes"], "tiempo_s": ls["tiempo"], "RP": RP, "EEV": EEV, "VSS": VSS,
           "reducido": {"estados": est_red, "L_shaped": ls_red["UB"], "equiv_det": de_red["obj"], "dif_rel": dif_red},
           "equiv_det_completo": de_full_tamano["obj"], "dif_equiv_det_completo": dif_de_full,
           "arbol": agg_sto, "regla_EV": agg_ev, "validacion": {"estocastica": fin_sto, "EV": fin_ev},
           "estados_sin_soporte": vacios, "respaldo_activado": respaldo_sto}
with open(os.path.join(DIR_RES, "resumen.json"), "w", encoding="utf-8") as fh:
    json.dump(resumen, fh, indent=2, ensure_ascii=False, default=float)
print("Archivos en resultados/:", sorted(os.listdir(DIR_RES)))

## 12. Empaquetar salidas

Comprime `datos/`, `resultados/` y `figuras/` para subirlos junto con el informe.

In [ ]:
with zipfile.ZipFile("practica2_salidas.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta in (DIR_DATOS, DIR_RES, DIR_FIG):
        for nombre in sorted(os.listdir(carpeta)):
            z.write(os.path.join(carpeta, nombre))
print("Listo: practica2_salidas.zip")
try:
    from google.colab import files
    files.download("practica2_salidas.zip")
except Exception:
    pass